## Fastaのアップロード

←のフォルダマークをクリックし、ファイルをアップロードするアイコンをクリック。

アップロードするFastaファイルを選んでアップロードする。


コムギなどあまりに大きい場合は、あらかじめ特定の染色体のみにするなど減らしておく。出現頻度を見るものなので、データを網羅する必要はない。


In [3]:
library(stringr)
fas <- scan("Osativa_204_v7.0.protein.fasta", what = character(), na.strings = "zzz" , sep = "\n")
genenames = fas[str_sub(fas, start = 1, end = 1) == ">"]
genes = data.frame(name = str_sub(genenames,start = 2,end = 12))
genes$anotation = str_sub(genenames,start = 13,end = -1)
fas[str_sub(fas, start = 1, end = 1) == ">"] = " "
allseq = paste(fas,sep = "",collapse = "")
seqs = str_split(allseq, " ")[[1]]
seqs <- seqs[2:length(seqs)]
genes$seq <- seqs

In [4]:
length(genes$seq)

[1] 49061

## バックグランドデータの作成。

実行時間はデータが大きいと数分になる。


dat1 <- c(dat1, substr(seq, j - 6, j + 6))の部分で取り出しを行っており、

上下６アミノ酸ずつ取り出すという意味になる。


        

In [5]:
library(parallel)
library(stringr)

cl <- makeCluster(detectCores() - 1)
clusterEvalQ(cl, library(stringr))

genes$seq2 <- parLapply(cl, genes$seq, function(seq) {
    if (str_length(seq) < 20) return(NA)
    dat1 <- c()
    for (j in 7:(str_length(seq) - 7)) {
        aa <- substr(seq, j, j)
        if (aa == "S" | aa == "T") {
            dat1 <- c(dat1, substr(seq, j - 6, j + 6))
        }
    }
    if (length(dat1) == 0) return(NA)
    paste(dat1, collapse = "\n")
})

stopCluster(cl)



[[1]]
[1] "stringr"   "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[7] "methods"   "base"

In [6]:
genes$seq2 <- unlist(genes$seq2)
writeLines(genes$seq2[!is.na(genes$seq2)], "BG_seq.txt")

←のファイル一覧にBG_seq.txtが出力されているはずなので、右クリックでダウンロード。
